# Sounds, Frequencies, and Patterns

Welcome! You are about to learn how to **see** sound.

By the end of this notebook, you will be able to look at a picture of any sound and
immediately understand what kind of sound it is — a pure musical note, a noisy rumble,
a rising tone, or a human voice. No experience needed at all.

We will listen to sounds, draw them on a screen, and look at them in clever ways.

**Put headphones in now if you have them — it makes a real difference!**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.signal import welch
from IPython.display import Audio, display

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

FS = 22050  # samples per second — the standard audio rate

def play(x, label=''):
    """Normalise and play a sound."""
    if label:
        print(label)
    normed = (x / (np.max(np.abs(x)) + 1e-10) * 0.9).astype(np.float32)
    display(Audio(normed, rate=FS))

def freq_picture(ax, x, title, xlim=(0, 5000), nperseg=None, color='C0'):
    """Plot how loud each frequency is (normalised to peak = 1)."""
    if nperseg is None:
        nperseg = min(len(x), FS)
    f, p = welch(x, fs=FS, nperseg=nperseg)
    p_norm = p / (p.max() + 1e-15)
    ax.fill_between(f, p_norm, alpha=0.55, color=color)
    ax.plot(f, p_norm, lw=1.2, color=color)
    ax.set_xlim(xlim); ax.set_ylim(0, 1.25)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Loudness')
    ax.set_title(title); ax.grid(alpha=0.3)

def sound_picture(ax, x, title, f_max=4000, vmin=-60, vmax=0):
    """Plot a spectrogram (time on x, frequency on y, brightness = loudness)."""
    f, ts, Sxx = signal.spectrogram(x, fs=FS, nperseg=1024, noverlap=768)
    db = 10 * np.log10(np.maximum(Sxx, 1e-12))
    ax.pcolormesh(ts, f, db, vmin=vmin, vmax=vmax, cmap='inferno', shading='gouraud')
    ax.set_ylim(0, f_max)
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Frequency (Hz)')
    ax.set_title(title)

print('Ready!')

---
## 1. What Is Sound?

When something vibrates — a guitar string, a speaker cone, your vocal cords — it pushes the
air around it back and forth. Those pushes travel outward through the air until they reach
your ear, where they make your eardrum vibrate too. Your brain turns that into what you
experience as sound.

We can draw a sound as a wiggly line. Time goes left to right, and the height of the line
shows how hard the air is pushing at each moment. When the line is high, the air is pushing
forward; when it is low, the air is pulling back.

The simplest possible sound is a **single pure tone** — one smooth, steady wave, like a
tuning fork or a held whistle note. Below is what that looks like.

In [ ]:
# A single pure tone at 440 Hz — the note A, used to tune instruments
DUR = 3.0
t3  = np.linspace(0, DUR, int(FS * DUR), endpoint=False)

tone_440 = np.sin(2 * np.pi * 440 * t3)

# Plot a short slice (0.01 s) so you can see the wiggle clearly
n_show = int(FS * 0.012)
fig, axes = plt.subplots(1, 2, figsize=(13, 3))

axes[0].plot(t3[:n_show] * 1000, tone_440[:n_show], lw=1.8, color='C0')
axes[0].set_xlabel('Time (milliseconds)')
axes[0].set_ylabel('Air pressure')
axes[0].set_title('Single pure tone — 440 Hz\nOne smooth, repeating wiggle')
axes[0].axhline(0, lw=0.6, color='k', alpha=0.3)

# Zoom out a little to see more cycles
n_show2 = int(FS * 0.05)
axes[1].plot(t3[:n_show2] * 1000, tone_440[:n_show2], lw=1.5, color='C0')
axes[1].set_xlabel('Time (milliseconds)')
axes[1].set_title('Same tone, slightly zoomed out\nEvery cycle looks identical')
axes[1].axhline(0, lw=0.6, color='k', alpha=0.3)

plt.tight_layout(); plt.show()

play(tone_440, 'A pure tone at 440 Hz:')

---
## 2. High and Low Frequencies

**Frequency** means "how many complete wiggles happen per second." We measure it in
**Hz** (short for Hertz). 440 Hz means 440 complete back-and-forth cycles every second.

- **Low frequency** → fewer wiggles per second → slow wave → sounds like **bass / rumble**
- **High frequency** → more wiggles per second → fast wave → sounds like **treble / squeak**

Below are four pure tones at very different frequencies, all drawn over the same tiny time
window. Watch how the waves go from slow and wide (low frequency) to fast and narrow
(high frequency) as you go down the plot.

In [ ]:
tone_60   = np.sin(2 * np.pi * 60   * t3)
tone_2k   = np.sin(2 * np.pi * 2000 * t3)
tone_4k   = np.sin(2 * np.pi * 4000 * t3)

freq_info = [
    (tone_60,  'C1', '60 Hz  — deep bass (slow wiggle)'),
    (tone_440, 'C0', '440 Hz — middle A'),
    (tone_2k,  'C2', '2 000 Hz — bright treble'),
    (tone_4k,  'C3', '4 000 Hz — high tone (fast wiggle)'),
]

t_short = np.linspace(0, 25e-3, int(FS * 25e-3), endpoint=False)

fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
for ax, (tone, color, lbl) in zip(axes, freq_info):
    ax.plot(t_short * 1e3, np.sin(2 * np.pi * int(lbl.split()[0].replace(',', '')) * t_short),
            color=color, lw=1.8)
    ax.axhline(0, lw=0.5, color='k', alpha=0.3)
    ax.set_yticklabels([])
    ax.annotate(lbl, xy=(0.01, 0.72), xycoords='axes fraction',
                fontsize=9, color=color, fontweight='bold')

axes[-1].set_xlabel('Time (milliseconds)')
fig.suptitle('Four pure tones (same 25 ms window)\n'
             'Low frequency = slow wiggle   |   High frequency = fast wiggle', fontsize=12)
plt.tight_layout(); plt.show()

print('Listen to each tone — notice how the pitch rises as the frequency increases:\n')
play(tone_60,  '60 Hz  (very deep — you may feel it more than hear it):')
play(tone_440, '440 Hz (clear musical note):')
play(tone_2k,  '2000 Hz (bright):')
play(tone_4k,  '4000 Hz (high):')

---
## 3. Mixing Sounds

Real sounds are almost never a single pure tone. Music, voices, engines, rain — they are
all mixtures of many frequencies happening at the same time.

Mixing sounds is simple: wherever two waves happen at the same moment, you just add their
heights together. If two waves both push up at the same instant, you get a bigger push up.
If one pushes up while the other pulls down, they partially cancel each other out.

Below: two different tones combined into one sound. You can see how the result is more
complex-looking than either wave alone — but it is simply the sum of the two.

In [ ]:
tone_220 = np.sin(2 * np.pi * 220 * t3)   # A3 — low A
tone_550 = np.sin(2 * np.pi * 550 * t3)   # about C#5 — bright
mixed_2  = tone_220 + tone_550

n_show = int(FS * 0.025)   # 25 ms slice
t_ms   = t3[:n_show] * 1000

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)

axes[0].plot(t_ms, tone_220[:n_show], lw=1.5, color='C0')
axes[0].set_title('Tone 1 — 220 Hz (slow wave)'); axes[0].set_yticklabels([]); axes[0].axhline(0, lw=0.4, color='k', alpha=0.3)

axes[1].plot(t_ms, tone_550[:n_show], lw=1.5, color='C1')
axes[1].set_title('Tone 2 — 550 Hz (fast wave)'); axes[1].set_yticklabels([]); axes[1].axhline(0, lw=0.4, color='k', alpha=0.3)

axes[2].plot(t_ms, mixed_2[:n_show], lw=1.5, color='C3')
axes[2].set_title('Both tones added together — more complex, but just the sum of the two above')
axes[2].set_yticklabels([]); axes[2].axhline(0, lw=0.4, color='k', alpha=0.3)
axes[2].set_xlabel('Time (milliseconds)')

fig.suptitle('Mixing two tones: add the heights at every moment', fontsize=12)
plt.tight_layout(); plt.show()

print('1) Tone 1 alone (220 Hz):')
play(tone_220)
print('2) Tone 2 alone (550 Hz):')
play(tone_550)
print('3) Both tones together:')
play(mixed_2)

In [ ]:
# A C-major chord: C4, E4, G4
tone_c4 = np.sin(2 * np.pi * 261.63 * t3)
tone_e4 = np.sin(2 * np.pi * 329.63 * t3)
tone_g4 = np.sin(2 * np.pi * 392.00 * t3)
chord   = tone_c4 + tone_e4 + tone_g4

n_show  = int(FS * 0.03)
t_ms    = t3[:n_show] * 1000
small   = {'lw': 1.2, 'alpha': 0.8}

fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True,
                          gridspec_kw={'height_ratios': [1, 1, 1, 2]})

for ax, tone, color, lbl in zip(
        axes[:3],
        [tone_c4, tone_e4, tone_g4],
        ['C0', 'C1', 'C2'],
        ['C4 — 262 Hz', 'E4 — 330 Hz', 'G4 — 392 Hz']):
    ax.plot(t_ms, tone[:n_show], color=color, **small)
    ax.set_title(lbl, fontsize=9); ax.set_yticklabels([]); ax.axhline(0, lw=0.4, color='k', alpha=0.3)

axes[3].plot(t_ms, chord[:n_show], lw=1.5, color='C4')
axes[3].set_title('All three combined — the C-major chord')
axes[3].set_yticklabels([]); axes[3].axhline(0, lw=0.4, color='k', alpha=0.3)
axes[3].set_xlabel('Time (milliseconds)')

fig.suptitle('Building a chord by adding three tones together', fontsize=12)
plt.tight_layout(); plt.show()

play(chord, 'C-major chord (three notes at once):')

---
## 4. The Frequency Picture

The wiggly time-trace is one way to draw a sound. But it has a problem: when many
frequencies are mixed together, the wiggly line looks complicated and it's hard to
tell what frequencies are inside it.

There's a smarter way to look at a sound:

> **Ask: how loud is each individual frequency?**

Imagine running a sound through a machine that, for each frequency from very low to very
high, measures "how much of this frequency is in here?" That gives you a bar chart —
frequency on the x-axis, loudness on the y-axis.

- A **single tone** at 440 Hz → one tall bar at 440 Hz, nothing anywhere else.
- A **chord** of three notes → three bars, one for each note.
- **All frequencies equally** → every bar the same height (we'll see this next).

This picture is called a **frequency fingerprint** of the sound.

In [ ]:
# Build longer signals for sharper frequency pictures
t5  = np.linspace(0, 5.0, int(FS * 5.0), endpoint=False)
s_tone  = np.sin(2 * np.pi * 440  * t5)
s_chord = (np.sin(2 * np.pi * 261.63 * t5) +
           np.sin(2 * np.pi * 329.63 * t5) +
           np.sin(2 * np.pi * 392.00 * t5))
s_noise = np.random.randn(len(t5))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

freq_picture(axes[0], s_tone,  'Single tone (440 Hz)\nOne spike — nothing else',
             xlim=(0, 1000), nperseg=FS, color='C0')
axes[0].axvline(440, color='red', ls='--', lw=1.5, alpha=0.7, label='440 Hz')
axes[0].legend(fontsize=9)

freq_picture(axes[1], s_chord, 'C-major chord (three notes)\nThree spikes, one per note',
             xlim=(0, 1000), nperseg=FS, color='C1')
for fv, lbl in [(261.63, 'C'), (329.63, 'E'), (392.00, 'G')]:
    axes[1].axvline(fv, color='red', ls='--', lw=1.3, alpha=0.7, label=f'{lbl} ({fv:.0f} Hz)')
axes[1].legend(fontsize=8)

freq_picture(axes[2], s_noise, 'White noise (coming up next)\nEvery frequency equally loud',
             xlim=(0, 5000), nperseg=FS // 8, color='C2')
axes[2].axhline(1.0, color='red', ls='--', lw=1.5, alpha=0.7, label='All equal')
axes[2].legend(fontsize=9)

fig.suptitle('Frequency fingerprint: how loud is each frequency?', fontsize=12)
plt.tight_layout(); plt.show()

---
## 5. All Frequencies at Once — Equal Mix

What if you mixed *every* frequency at the same time, all equally loud?

You would get a hiss — a continuous, featureless shhhh with no pitch, no tone,
no pattern at all.

This is called **white noise**, by analogy with white light.
White light contains all colours of the visible spectrum equally.
White noise contains all audio frequencies equally.

**First:** we will hear white noise across all frequencies.

**Then:** we will hear white noise that has been cut to include only frequencies between
500 Hz and 2000 Hz — still equal within that range, but nothing below 500 or above 2000.
Notice how it changes the character of the hiss.

In [ ]:
t4   = np.linspace(0, 4.0, int(FS * 4.0), endpoint=False)
N4   = len(t4)

# Full white noise — all frequencies, all equal
white_noise = np.random.randn(N4)

# Bandlimited noise — only 500–2000 Hz, equally loud within that band
bn_fd    = np.fft.rfft(np.random.randn(N4))
bn_freqs = np.fft.rfftfreq(N4, 1.0 / FS)
bn_fd[bn_freqs < 500]  = 0.0
bn_fd[bn_freqs > 2000] = 0.0
band_noise = np.fft.irfft(bn_fd, n=N4)
band_noise /= np.std(band_noise)

fig, axes = plt.subplots(2, 3, figsize=(15, 6))

# White noise row
n_show = int(FS * 0.05)
axes[0, 0].plot(t4[:n_show] * 1000, white_noise[:n_show], lw=0.5)
axes[0, 0].set_xlabel('Time (ms)'); axes[0, 0].set_ylabel('Amplitude')
axes[0, 0].set_title('White noise — time trace\nCompletely random, no pattern')

freq_picture(axes[0, 1], white_noise, 'White noise — frequency fingerprint\nFlat: every frequency equal',
             xlim=(0, 5000), nperseg=FS // 8, color='C0')
axes[0, 1].axhline(1.0, color='red', ls='--', lw=1.5, alpha=0.6, label='Equal level')
axes[0, 1].legend(fontsize=9)

f_sp, ts, Sxx = signal.spectrogram(white_noise, fs=FS, nperseg=512, noverlap=384)
db = 10 * np.log10(np.maximum(Sxx, 1e-12))
axes[0, 2].pcolormesh(ts, f_sp, db, vmin=-50, vmax=10, cmap='inferno', shading='gouraud')
axes[0, 2].set_ylim(0, 5000); axes[0, 2].set_xlabel('Time (s)'); axes[0, 2].set_ylabel('Frequency (Hz)')
axes[0, 2].set_title('White noise — sound picture\nUniformly bright everywhere')

# Bandlimited noise row
axes[1, 0].plot(t4[:n_show] * 1000, band_noise[:n_show], lw=0.5, color='C1')
axes[1, 0].set_xlabel('Time (ms)'); axes[1, 0].set_ylabel('Amplitude')
axes[1, 0].set_title('Band noise (500–2000 Hz) — time trace\nStill random, but smoother')

freq_picture(axes[1, 1], band_noise,
             'Band noise — frequency fingerprint\nFlat between 500–2000 Hz, empty elsewhere',
             xlim=(0, 5000), nperseg=FS // 4, color='C1')
axes[1, 1].axvspan(500, 2000, alpha=0.08, color='C1', label='Active band')
axes[1, 1].legend(fontsize=9)

f_sp2, ts2, Sxx2 = signal.spectrogram(band_noise, fs=FS, nperseg=512, noverlap=384)
db2 = 10 * np.log10(np.maximum(Sxx2, 1e-12))
axes[1, 2].pcolormesh(ts2, f_sp2, db2, vmin=-50, vmax=10, cmap='inferno', shading='gouraud')
axes[1, 2].set_ylim(0, 5000); axes[1, 2].set_xlabel('Time (s)'); axes[1, 2].set_ylabel('Frequency (Hz)')
axes[1, 2].set_title('Band noise — sound picture\nBright only between 500–2000 Hz')

fig.suptitle('Equal mix within a range — all bars the same height inside the chosen band', fontsize=12)
plt.tight_layout(); plt.show()

play(white_noise, 'White noise (all frequencies):')
play(band_noise,  'Band noise (500–2000 Hz only):')

---
## 6. Unequal Mix — Some Frequencies Louder

What if the low frequencies are much, much louder than the high ones?

You get a heavy bass rumble — the kind of sound you feel in your chest at a concert,
or the low throb of a passing lorry. The high frequencies are still there, but the
bass overwhelms them completely.

In the frequency fingerprint, instead of flat bars, you see bars that are tall on the left
(low frequencies) and shrink down to the right (high frequencies). The bars form a slope.

This kind of uneven noise — where some frequencies are much louder than others — is called
**coloured noise**. (The "colour" is an analogy with coloured light, which lacks some colours
that white light would have equally.)

Real-world noise is almost always coloured. Rumbles from traffic, wind, machinery — they all
have this same shape: loud at low frequencies, quieter at high frequencies.

In [ ]:
# Make bass-heavy noise: start with white noise, make low frequencies much louder
raw      = np.random.randn(N4)
raw_fd   = np.fft.rfft(raw)
freqs_fd = np.fft.rfftfreq(N4, 1.0 / FS)
freqs_fd[0] = 1.0   # avoid dividing by zero

# Multiply each frequency by 1/f — low frequencies get amplified, high ones don't
boost     = 60.0 / np.maximum(freqs_fd, 1.0)
col_noise = np.fft.irfft(raw_fd * boost, n=N4)
col_noise /= np.std(col_noise)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

n_show = int(FS * 0.05)
axes[0].plot(t4[:n_show] * 1000, col_noise[:n_show], lw=0.7, color='C1')
axes[0].set_xlabel('Time (ms)'); axes[0].set_ylabel('Amplitude')
axes[0].set_title('Coloured noise — time trace\nLarge slow swings dominate')

freq_picture(axes[1], col_noise, 'Coloured noise — frequency fingerprint\nHigh on the left, low on the right',
             xlim=(20, 5000), nperseg=FS // 8, color='C1')
# Overlay white noise for comparison
f_wn, p_wn = welch(white_noise, fs=FS, nperseg=FS // 8)
p_wn_norm  = p_wn / p_wn.max()
axes[1].plot(f_wn, p_wn_norm, lw=1.2, color='C0', ls='--', alpha=0.7, label='White noise (flat)')
axes[1].set_xscale('log'); axes[1].legend(fontsize=9)
axes[1].set_xlim(20, 5000)

f_sp3, ts3, Sxx3 = signal.spectrogram(col_noise, fs=FS, nperseg=512, noverlap=384)
db3 = 10 * np.log10(np.maximum(Sxx3, 1e-12))
axes[2].pcolormesh(ts3, f_sp3, db3, vmin=-30, vmax=30, cmap='inferno', shading='gouraud')
axes[2].set_ylim(0, 5000); axes[2].set_xlabel('Time (s)'); axes[2].set_ylabel('Frequency (Hz)')
axes[2].set_title('Coloured noise — sound picture\nBright at bottom, dark at top')

fig.suptitle('Coloured noise: low frequencies are much louder than high ones', fontsize=12)
plt.tight_layout(); plt.show()

play(col_noise, 'Coloured noise — a heavy bass rumble:')

---
## 7. Frequencies That Change Over Time — The Chirp

All the sounds so far had the same frequencies all the way through.
But what if the frequency *changes* as time passes?

A **chirp** is a sound where the frequency rises (or falls) continuously over time.
You have definitely heard chirps before:

- A bird call that starts low and sweeps up
- A siren as a car accelerates away from you
- A slide whistle going upward
- Dolphins and bats use chirps to navigate (echolocation)

In a chirp, the wave looks normal at first — then it gradually gets squished tighter and
tighter as the wiggles speed up. Below, watch how the wave changes as you move right.

In [ ]:
# A chirp: starts at 80 Hz and rises to 2000 Hz over 5 seconds
t5c = np.linspace(0, 5.0, int(FS * 5.0), endpoint=False)
chirp_sig = signal.chirp(t5c, f0=80, f1=2000, t1=5.0, method='quadratic')
ramp = int(0.05 * FS)
chirp_sig[:ramp]  *= np.linspace(0, 1, ramp)
chirp_sig[-ramp:] *= np.linspace(1, 0, ramp)

# Show the wave at three moments: start, middle, end
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

windows = [(0.3, 'Start  (t ≈ 0.3 s)\nSlow wiggle — low frequency'),
           (2.5, 'Middle (t ≈ 2.5 s)\nMedium speed'),
           (4.7, 'End    (t ≈ 4.7 s)\nFast wiggle — high frequency')]

win_dur = 0.04   # 40 ms window
for ax, (t_cen, lbl) in zip(axes, windows):
    start = max(0, int(t_cen * FS) - int(win_dur * FS // 2))
    end   = start + int(win_dur * FS)
    t_win = np.linspace(0, win_dur * 1000, end - start)
    ax.plot(t_win, chirp_sig[start:end], lw=1.5, color='C2')
    ax.axhline(0, lw=0.4, color='k', alpha=0.3)
    ax.set_title(lbl, fontsize=9); ax.set_xlabel('Time (ms)'); ax.set_yticklabels([])

fig.suptitle('A chirp: the same wave, but getting faster and faster over time', fontsize=12)
plt.tight_layout(); plt.show()

play(chirp_sig, 'The chirp — a rising tone (80 Hz → 2000 Hz over 5 seconds):')

---
## 8. What Does Speech Look Like?

When you speak, two things happen at the same time:

1. **Your voice box vibrates** at a steady rate — about 100 to 200 times a second for
   most people. This creates a set of tones that are evenly spaced in frequency: 130 Hz,
   260 Hz, 390 Hz, 520 Hz, and so on. These are called **harmonics**.

2. **Your mouth and throat shape those tones.** The space inside your mouth acts like a
   room with its own resonances. Depending on the shape of your lips, tongue, and jaw,
   certain harmonics get boosted and others get quieted. The boosted frequency ranges are
   what make each vowel sound distinct.

That is why "ahh" and "ee" sound different even at the same pitch — the mouth shape
changes, which changes which harmonics get boosted.

Below we synthesise two vowel sounds and listen to them, then look at what they look like
in a sound picture. See if you can spot the difference.

In [ ]:
def synth_vowel(f0, formants, dur):
    """Build a vowel from harmonics, boosted near the formant frequencies."""
    t_v = np.linspace(0, dur, int(FS * dur), endpoint=False)
    y   = np.zeros(len(t_v))
    k   = 1
    while True:
        freq = k * f0
        if freq > 5000:
            break
        # How much to include this harmonic — depends on how close it is to the formants
        amp = sum(np.exp(-((freq - fi) / bw) ** 2) for fi, bw in formants)
        y  += amp * np.sin(2 * np.pi * freq * t_v)
        k  += 1
    return y / (np.max(np.abs(y)) + 1e-10)

f0 = 150   # pitch (voice box vibration rate)

# Vowel "ahh": main resonances at about 750 Hz and 1100 Hz
ah = synth_vowel(f0, formants=[(750, 100), (1100, 130), (2500, 200)], dur=1.5)

# Vowel "ee": resonances shift — low one drops, high one rises
ee = synth_vowel(f0, formants=[(280, 80),  (2400, 180), (3000, 220)], dur=1.5)

# Put them together with a short gap
silence = np.zeros(int(FS * 0.3))
speech  = np.concatenate([ah, silence, ee])
t_sp    = np.linspace(0, len(speech) / FS, len(speech), endpoint=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Time traces side by side
n_short = int(FS * 0.03)
axes[0].plot(np.linspace(0, 30, n_short), ah[:n_short],  lw=1.2, color='C4', label='"ahh"')
axes[0].plot(np.linspace(0, 30, n_short), ee[:n_short],  lw=1.2, color='C5', label='"ee"',  alpha=0.85)
axes[0].set_xlabel('Time (ms)'); axes[0].set_yticklabels([])
axes[0].set_title('Time trace — both vowels\nSame pitch, different shape')
axes[0].legend(fontsize=9)

# Frequency fingerprints
t_long = np.linspace(0, 5.0, int(FS * 5.0), endpoint=False)
ah5    = synth_vowel(f0, [(750, 100), (1100, 130), (2500, 200)], 5.0)
ee5    = synth_vowel(f0, [(280, 80),  (2400, 180), (3000, 220)], 5.0)
f_ah, p_ah = welch(ah5, fs=FS, nperseg=FS)
f_ee, p_ee = welch(ee5, fs=FS, nperseg=FS)
axes[1].plot(f_ah, p_ah / p_ah.max(), lw=1.5, color='C4', label='"ahh"')
axes[1].plot(f_ee, p_ee / p_ee.max(), lw=1.5, color='C5', label='"ee"', alpha=0.85)
axes[1].set_xlim(0, 4000); axes[1].set_xlabel('Frequency (Hz)'); axes[1].set_ylabel('Loudness')
axes[1].set_title('Frequency fingerprint\nThe boosted ranges are in different places')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

# Sound picture of the full sequence
f_sg, ts_sg, Sxx_sg = signal.spectrogram(speech, fs=FS, nperseg=1024, noverlap=900)
db_sg = 10 * np.log10(np.maximum(Sxx_sg, 1e-12))
axes[2].pcolormesh(ts_sg, f_sg, db_sg, vmin=-50, vmax=0, cmap='inferno', shading='gouraud')
axes[2].set_ylim(0, 4000); axes[2].set_xlabel('Time (s)'); axes[2].set_ylabel('Frequency (Hz)')
axes[2].set_title('"ahh" then "ee" — sound picture\nBright bands shift between vowels')
axes[2].axvline(1.5, color='white', ls='--', lw=1.2, alpha=0.7)
axes[2].text(0.75, 3700, '"ahh"', color='white', ha='center', fontsize=10)
axes[2].text(2.15, 3700, '"ee"',  color='white', ha='center', fontsize=10)

fig.suptitle('Two vowels: same pitch, different mouth shape → different frequency pattern', fontsize=12)
plt.tight_layout(); plt.show()

play(ah,     '"ahh" vowel:')
play(ee,     '"ee" vowel:')
play(speech, '"ahh" then "ee":')

---
## 9. The Sound Picture

We have now seen two ways to look at a sound:

| View | What it shows | What it hides |
|---|---|---|
| **Time trace** | How the wave moves moment by moment | Which frequencies are present |
| **Frequency fingerprint** | Which frequencies are present and how loud | When things happen |

What we really want is **both at once**: what frequencies are present *and* when they happen.

### How it is made

1. **Chop** the sound into many short overlapping windows (each about 50 ms long).
2. For each window, make a **frequency fingerprint**.
3. **Stack** all those fingerprints side by side, from left (beginning) to right (end).
4. Use **brightness** instead of bar height to show loudness — bright means loud, dark means quiet.

The result is called a **spectrogram** (or *sound picture* here). It shows:
- **Left to right** = time (early → late)
- **Bottom to top** = frequency (low → high)
- **Bright spots** = loud at that frequency at that moment

Below: watch how the chirp from before turns into a rising diagonal line in the sound picture.

In [ ]:
# Show 3 time windows of the chirp, their frequency fingerprints, and the full picture
window_dur = 0.25   # seconds
window_n   = int(FS * window_dur)
centers    = [0.5, 2.5, 4.5]
c_labels   = ['Early (t ≈ 0.5 s)\n→ slow wave, low freq',
               'Middle (t ≈ 2.5 s)\n→ medium speed',
               'Late (t ≈ 4.5 s)\n→ fast wave, high freq']

fig = plt.figure(figsize=(15, 9))
gs  = fig.add_gridspec(3, 3, hspace=0.55, wspace=0.35)

for col, (tc, lbl) in enumerate(zip(centers, c_labels)):
    # --- Top row: time slice ---
    ax_t = fig.add_subplot(gs[0, col])
    start = max(0, int(tc * FS) - window_n // 2)
    end   = min(len(chirp_sig), start + window_n)
    t_win = np.linspace(0, window_dur * 1000, end - start)
    ax_t.plot(t_win, chirp_sig[start:end], lw=1.2, color='C2')
    ax_t.set_title(lbl, fontsize=9); ax_t.set_xlabel('Time (ms)'); ax_t.set_yticklabels([])
    ax_t.axhline(0, lw=0.4, color='k', alpha=0.3)

    # --- Middle row: frequency fingerprint of that slice ---
    ax_f = fig.add_subplot(gs[1, col])
    f_sl, p_sl = welch(chirp_sig[start:end], fs=FS, nperseg=window_n // 2)
    ax_f.fill_between(f_sl, p_sl / (p_sl.max() + 1e-15), alpha=0.6, color='C2')
    ax_f.plot(f_sl, p_sl / (p_sl.max() + 1e-15), lw=1.2, color='C2')
    peak_f = f_sl[np.argmax(p_sl)]
    ax_f.axvline(peak_f, color='red', ls='--', lw=1.5, alpha=0.8, label=f'Peak: {peak_f:.0f} Hz')
    ax_f.set_xlim(0, 3000); ax_f.set_ylim(0, 1.3)
    ax_f.set_xlabel('Frequency (Hz)'); ax_f.set_ylabel('Loudness')
    ax_f.set_title(f'Frequency fingerprint\nPeak at {peak_f:.0f} Hz', fontsize=9)
    ax_f.legend(fontsize=8); ax_f.grid(alpha=0.3)

# --- Bottom: full spectrogram ---
ax_sg = fig.add_subplot(gs[2, :])
f_sg2, ts_sg2, Sxx2 = signal.spectrogram(chirp_sig, fs=FS, nperseg=1024, noverlap=900)
db_sg2 = 10 * np.log10(np.maximum(Sxx2, 1e-12))
ax_sg.pcolormesh(ts_sg2, f_sg2, db_sg2, vmin=-60, vmax=0, cmap='inferno', shading='gouraud')
ax_sg.set_ylim(0, 3000); ax_sg.set_xlabel('Time (s)'); ax_sg.set_ylabel('Frequency (Hz)')
ax_sg.set_title('Complete sound picture: all windows stacked — the rising diagonal is the chirp')
for tc in centers:
    ax_sg.axvline(tc, color='white', ls='--', lw=1.2, alpha=0.7)

fig.suptitle('Building a sound picture: slice → fingerprint → stack',
             fontsize=13, y=1.01)
plt.show()

In [ ]:
# All the sounds we have heard, shown as sound pictures side by side
t4b = np.linspace(0, 4.0, int(FS * 4.0), endpoint=False)

# Regenerate the sounds we need at 4s duration for consistency
s_440    = np.sin(2 * np.pi * 440 * t4b)
s_chord4 = (np.sin(2 * np.pi * 261.63 * t4b) +
            np.sin(2 * np.pi * 329.63 * t4b) +
            np.sin(2 * np.pi * 392.00 * t4b))

# Chirp already generated at 5 s — trim or use as-is (will have different duration label)
# Make a 4-second chirp for visual consistency
t4c = np.linspace(0, 4.0, int(FS * 4.0), endpoint=False)
chirp4 = signal.chirp(t4c, f0=80, f1=2000, t1=4.0, method='quadratic')

# White noise and coloured noise already at 4 s
# Speech — 4 seconds total: ahh + gap + ee
ah4  = synth_vowel(150, [(750, 100), (1100, 130), (2500, 200)], 1.7)
ee4  = synth_vowel(150, [(280, 80),  (2400, 180), (3000, 220)], 1.7)
sp4  = np.concatenate([ah4, np.zeros(int(FS * 0.6)), ee4])

sounds = [
    (s_440,      'Pure tone (440 Hz)\nHorizontal line at one frequency'),
    (s_chord4,   'Chord (3 notes)\nThree horizontal lines'),
    (chirp4,     'Chirp (rising tone)\nRising diagonal'),
    (white_noise[:len(t4b)], 'White noise\nUniformly bright everywhere'),
    (col_noise[:len(t4b)],   'Coloured noise\nBright at bottom, fading upward'),
    (sp4,        'Vowels ("ahh" then "ee")\nBands shift as the vowel changes'),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, (s, title) in zip(axes.flat, sounds):
    sound_picture(ax, s, title, f_max=4000, vmin=-60, vmax=0)

fig.suptitle('The Grand Tour — Six Sounds as Pictures\n'
             'Can you see the features we talked about in each one?', fontsize=13)
plt.tight_layout()
plt.show()

---
## You Made It!

Look at what you can now do:

- Draw a sound as a wiggly line and say what the shape means
- Tell the difference between a low-frequency wave and a high-frequency wave just by looking
- Understand that any sound is a mix of different frequencies added together
- Recognise a flat frequency fingerprint (equal mix) and a sloped one (unequal mix)
- Spot a chirp — a rising tone — in both the time trace and the sound picture
- Read a sound picture (spectrogram) and extract information from it

**That is a genuinely powerful set of skills.** Scientists, engineers, and doctors use
exactly these ideas every day to understand signals — everything from music to medical scans
to the deep rumble of distant earthquakes.

---

### What's Next?

In the next notebook, we explore a tricky problem: what happens when a signal you care about
gets completely buried inside noise that is much louder at certain frequencies?

And we will learn one clever trick that makes the hidden signal reappear — almost like magic.

See you there!